<a href="https://colab.research.google.com/github/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/blob/main/notebooks/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [80]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import string
import re
import html

# PyTorch (Required for BERT)
# import torch

# NLTK Libraries
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# NLTK Preprocessing Tools
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Train-Test Split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Text Vectorization (Bag of Words & TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classical Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Model Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report)

# BERT Tokenizer and Pre-trained Model
# from transformers import AutoTokenizer
# from transformers import AutoModel

# Deep learning models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# Used to save and load trained machine learning models
import joblib
import os
import pickle

# Used to save and load trained deep learning models
from pathlib import Path
from google.colab import files

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Read the Data

In [2]:
# URL of the dataset stored in the GitHub repository
filepath = "https://raw.githubusercontent.com/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/main/data/Recipe%20Reviews%20and%20User%20Feedback%20Dataset.csv"

# Load the dataset
df_review = pd.read_csv(filepath)

# Display the first 5 rows
df_review.head()

,Unnamed: 0,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score,text
0,0,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM,u_9iFLIhMa8QaG,Jeri326,1,1665619889,0,0,0,5,527,"I tweaked it a little, removed onions because ..."
1,1,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY,u_Lu6p25tmE77j,Mark467,50,1665277687,0,7,0,5,724,Bush used to have a white chili bean and it ma...
2,2,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FPrSGyTv7PQkZq37j92r9mYGkP,u_s0LwgpZ8Jsqq,Barbara566,10,1664404557,0,3,0,5,710,I have a very complicated white chicken chili ...
3,3,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DzdSIgV9qNiuBaLoZ7JQaartoC,u_fqrybAdYjgjG,jeansch123,1,1661787808,2,2,0,0,581,"In your introduction, you mentioned cream chee..."
4,4,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DtZJuRQYeTFwXBoZRfRhBPEXjI,u_XXWKwVhKZD69,camper77,10,1664913823,1,7,0,0,820,Wonderful! I made this for a &#34;Chili/Stew&#...


# EDA

In [3]:
# Identifying numerical and categorical data

df_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18182 entries, 0 to 18181
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       18182 non-null  int64 
 1   recipe_number    18182 non-null  int64 
 2   recipe_code      18182 non-null  int64 
 3   recipe_name      18182 non-null  object
 4   comment_id       18182 non-null  object
 5   user_id          18182 non-null  object
 6   user_name        18182 non-null  object
 7   user_reputation  18182 non-null  int64 
 8   created_at       18182 non-null  int64 
 9   reply_count      18182 non-null  int64 
 10  thumbs_up        18182 non-null  int64 
 11  thumbs_down      18182 non-null  int64 
 12  stars            18182 non-null  int64 
 13  best_score       18182 non-null  int64 
 14  text             18180 non-null  object
dtypes: int64(10), object(5)
memory usage: 2.1+ MB


In [4]:
# Identifying dataset size

df_review.shape

(18182, 15)

In [5]:
# Statistical Summary

df_review.describe()

,Unnamed: 0,recipe_number,recipe_code,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score
count,18182.000000,18182.000000,18182.000000,18182.000000,1.818200e+04,18182.000000,18182.000000,18182.000000,18182.000000,18182.000000
mean,121.465295,38.689363,21773.667253,2.159608,1.623710e+09,0.014630,1.089264,0.549335,4.288802,153.162138
std,116.747893,29.786647,23965.109637,10.014666,5.468697e+06,0.137974,4.201004,3.470124,1.544786,141.075316
min,0.000000,1.000000,386.000000,0.000000,1.613035e+09,0.000000,0.000000,0.000000,0.000000,0.000000
25%,45.000000,12.000000,6086.000000,1.000000,1.622717e+09,0.000000,0.000000,0.000000,5.000000,100.000000
50%,91.000000,33.000000,14600.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
75%,150.000000,64.000000,33121.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
max,724.000000,100.000000,191775.000000,520.000000,1.665756e+09,3.000000,106.000000,126.000000,5.000000,946.000000


In [6]:
# Target Variable Analysis

df_review["stars"].value_counts()

# 1–5 are actual rating classes.
# 0 is not a rating, it means the user did not provide a rating.

,count
stars,
5,13829
0,1696
4,1655
3,490
1,280
2,232


In [7]:
# Trying to findout if there is any missing value present

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [8]:
# Handling the missing value

df_review.dropna(subset=["text"], inplace=True)

In [9]:
# After handling the misssing values

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [10]:
# Duplicate Rows

df_review.duplicated().sum()

np.int64(0)

In [11]:
# Count the number of unique reviews in the 'text' column

df_review["text"].nunique()

17731

In [12]:
# Duplicate review text

df_review["text"].duplicated().sum()

np.int64(449)

In [13]:
#  Inspect them

df_review[df_review["text"].duplicated(keep=False)].sort_values("text")

,Unnamed: 0,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score,text
252,252,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_107272,u_1oKVaAf9Tks6DDSWzXwN6Q5WKBN,ziki01,1,1622716893,0,0,0,5,100,.
6016,48,17,36450,Fluffy Key Lime Pie,sp_aUSaElGf_36450_c_107441,u_1oKVZb1mFfcT6ORS5FagojvK7rH,GStaelens,1,1622716887,0,0,1,3,100,.
167,167,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_107293,u_1oKVaKbaDb9XpuJSTzLKD4ult94,taylor,1,1622716893,0,0,0,5,100,.
4284,104,11,12003,Traditional Lasagna,sp_aUSaElGf_12003_c_107332,u_1oKVZb1mFfcT6ORS5FagojvK7rH,GStaelens,1,1622716881,0,0,0,5,100,.
16064,88,82,18274,Ravioli Lasagna,sp_aUSaElGf_18274_c_107642,u_1oKVZpBH6PLJXY2WdzR0I1YxFlr,smvlpn,1,1622716887,0,0,0,4,100,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6332,138,18,2872,Stuffed Pepper Soup,sp_aUSaElGf_2872_c_378868,u_1oKb4IFxLy8R9TkSJiPthlA5aj1,Lisabarney671,1,1622718230,0,0,0,5,100,yummy
10050,150,38,1063,Frosted Banana Bars,sp_aUSaElGf_1063_c_106240,u_1oKVZlbJsKnks4yML21aFN0AsFC,saw-whet,1,1622716880,0,0,0,0,100,
15049,122,74,26937,Pineapple Pudding Cake,sp_aUSaElGf_26937_c_106837,u_1oKVeA3xzAOfmCXeDrITZ2mGGIR,conshanty,1,1622716881,0,0,0,5,100,
5693,247,15,10252,Li’l Cheddar Meat Loaves,sp_aUSaElGf_10252_c_107176,u_1oKVZeTXJSMyvTZ0ijow6MBp0zM,ScottsdalePrincess,1,1622716888,0,0,0,0,100,My son loves these. I have made them many man...


In [14]:
# Remove duplicate reviews based on the 'text' column
# Keep only the first occurrence of each review
# Reset the index after removing duplicates

df_review = df_review.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

In [15]:
# Checking the review text

df_review["text"].duplicated().sum()

np.int64(0)

In [16]:
# removing unwanted colums
df_review.drop(columns = ['Unnamed: 0','recipe_number','recipe_code','comment_id','user_id','user_reputation','created_at','reply_count','best_score'],inplace = True)

In [17]:
df_review.drop(columns = ['thumbs_up','thumbs_down','user_name'],inplace = True)

# NLP Preprocessing

## Lowercasing

In [18]:
# Convert to lowercase

df_review['recipe_name'] = df_review['recipe_name'].str.lower()
df_review['text'] = df_review['text'].astype(str).str.lower()

## Punctuation

In [19]:
# Remove punctuation

text_columns = ["recipe_name", "text"]

for col in text_columns:
    df_review[col] = df_review[col].fillna("").str.lower()
    df_review[col] = df_review[col].str.translate(
        str.maketrans("", "", string.punctuation))

## URL

In [20]:
# Checking for the URLs

url_count = df_review["text"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in text: {url_count}")

url_count = df_review["recipe_name"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in recipe name: {url_count}")

Reviews containing URLs in text: 34
Reviews containing URLs in recipe name: 0


In [21]:
# Remove URLs

df_review["text"] = df_review["text"].str.replace(r"http\S+|www\.\S+", "", regex=True)

## HTML

In [22]:
# Convert HTML entities to normal characters and ensure text is in string format

df_review["text"] = df_review["text"].apply(
    lambda x: html.unescape(str(x))
)

# Emoji

In [23]:
# Checking for emojis

# Compile emoji pattern once

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE)

# Check multiple columns
text_columns = ["text", "recipe_name"]

for col in text_columns:
    emoji_count = df_review[col].apply(
        lambda x: bool(emoji_pattern.search(str(x)))).sum()

    print(f"Rows containing emojis in '{col}': {emoji_count}")

Rows containing emojis in 'text': 11
Rows containing emojis in 'recipe_name': 0


In [24]:
# Removing the emojis

df_review["text"] = df_review["text"].apply(
    lambda x: emoji_pattern.sub("", str(x)))

## Numbers

In [25]:
# Checking for numbers

text_columns = ["text", "recipe_name"]

for col in text_columns:
    number_count = df_review[col].str.contains(r"\d", regex=True, na=False).sum()
    print(f"Rows containing numbers in '{col}': {number_count}")

Rows containing numbers in 'text': 8776
Rows containing numbers in 'recipe_name': 0


In [26]:
# Display rows containing numbers

number_rows = df_review[df_review["text"].str.contains(r"\d", regex=True, na=False)]
number_rows["text"].head(20)

,text
4,wonderful i made this for a 34chilistew34 nigh...
6,wow this recipe is excellent as written the ...
7,this is delicious and i make it often one such...
8,i absolutely love this recipe i39ve tweaked it...
11,best white chili recipe i39ve had i served it ...
12,this recipe was excellent i added the cream ch...
14,fantastic but mild i added half a carolina rea...
21,this is our goto chicken chili recipe and has ...
23,this is just white chicken chili with i first ...
24,wow total wow totally delicious 5 stars plus


In [27]:
# Convert the 'text' column to string data type
df_review["text"] = df_review["text"].astype("string")

# Check the data type of the 'text' column
print(df_review["text"].dtype)

string


In [28]:
# Convert star ratings into sentiment labels
# 1–2 stars → Negative sentiment
# 3 stars   → Neutral sentiment
# 4–5 stars → Positive sentiment

def create_sentiment(stars):
    if stars in [1, 2]:
        return "Negative"
    elif stars == 3:
        return "Neutral"
    elif stars in [4, 5]:
        return "Positive"
    return None


# Create a new 'sentiment' column based on the star ratings
df_review["sentiment"] = df_review["stars"].apply(create_sentiment)

# Remove reviews with no valid sentiment label, such as unrated reviews (0 stars)
df_review = df_review.dropna(subset=["sentiment"]).copy()

In [29]:
# Count the number of reviews in each sentiment category
# This helps check the distribution of Negative, Neutral, and Positive reviews

df_review["sentiment"].value_counts()

,count
sentiment,
Positive,15107
Negative,509
Neutral,476


## Tokenization

In [30]:
text = df_review['text'].apply(word_tokenize)

## Stopword Removal

In [31]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  tokens = word_tokenize(text)
  filtered = [word for word in tokens if word not in stop_words]
  return " ".join(filtered)

df_review['text'] = df_review['text'].apply(remove_stopwords)

## Lemmatization

In [32]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
  tokens = word_tokenize(text)
  lemmas = [lemmatizer.lemmatize(word) for word in tokens]
  return " ".join(lemmas)

df_review['text'] = df_review['text'].apply(lemmatize_text)

# Classical Machine Learning Models

In [33]:
# Define the review text as the input feature

X = df_review["text"]

# Use sentiment as the target variable
# Negative, Neutral, Positive

y = df_review["sentiment"]

### Train-Test Split

In [34]:
# Input feature
X = df_review["text"].fillna("")

# Target variable
y = df_review["sentiment"]

# Convert sentiment labels into numbers
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# Check the mapping
print("Classes:", label_encoder.classes_)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# The vectorizer is fitted only on the training data to prevent data leakage.
# The learned vocabulary is then used to transform both the training and test data.

Classes: ['Negative' 'Neutral' 'Positive']


In [35]:
# Verify the shapes

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_train dtype:", y_train.dtype)
print("y_test :", y_test.shape)

X_train: (12873,)
X_test : (3219,)
y_train shape: (12873,)
y_train dtype: int64
y_test : (3219,)


# Text Vectorization

## Bag of Words

In [36]:
# Bag of Words

bow = CountVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# we are checking for the word counts

In [37]:
# Checking the Output

print("Training Shape :", X_train_bow.shape)
print("Testing Shape  :", X_test_bow.shape)

print("Vocabulary Size :", len(bow.vocabulary_))

Training Shape : (12873, 5000)
Testing Shape  : (3219, 5000)
Vocabulary Size : 5000


In [38]:
# View Vocabulary

feature_names = bow.get_feature_names_out()
print(feature_names[:50])

['10' '10 min' '10 minute' '10 star' '10 year' '100' '11' '112' '112 cup'
 '12' '12 cup' '12 hour' '12 lb' '12 oz' '12 pound' '12 recipe'
 '12 teaspoon' '12 tsp' '12 year' '125' '12c' '13' '13 cup' '13 pan'
 '13x9' '14' '14 cup' '14 oz' '14 teaspoon' '14 tsp' '15' '15 cup'
 '15 min' '15 minute' '15 year' '15x10' '16' '16 oz' '18' '18 tsp' '19'
 '1lb' '1st' '1st time' '1tsp' '20' '20 min' '20 minute' '20 year' '23']


In [39]:
# Evaluation Function

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred,
        average="weighted")

    recall = recall_score(y_test, y_pred,
        average="weighted")

    f1 = f1_score(y_test, y_pred,
        average="weighted")

    print("Accuracy :", round(accuracy,4))
    print("Precision:", round(precision,4))
    print("Recall   :", round(recall,4))
    print("F1 Score :", round(f1,4))

    print("\nClassification Report\n")
    print(classification_report(y_test,y_pred))

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_test,y_pred))

    return accuracy, precision, recall, f1

In [40]:
# Creating Results List

results = []

In [41]:
# Train Logistic Regression

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42)

lr.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.9056
Precision: 0.9346
Recall   : 0.9056
F1 Score : 0.9185

Classification Report

              precision    recall  f1-score   support

           0       0.31      0.47      0.38       102
           1       0.19      0.35      0.25        95
           2       0.98      0.94      0.96      3022

    accuracy                           0.91      3219
   macro avg       0.49      0.59      0.53      3219
weighted avg       0.93      0.91      0.92      3219


Confusion Matrix

[[  48   30   24]
 [  25   33   37]
 [  81  107 2834]]


In [42]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    class_weight="balanced",
    random_state=42)

svm.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.9205
Precision: 0.9302
Recall   : 0.9205
F1 Score : 0.925

Classification Report

              precision    recall  f1-score   support

           0       0.36      0.48      0.41       102
           1       0.22      0.25      0.24        95
           2       0.97      0.96      0.96      3022

    accuracy                           0.92      3219
   macro avg       0.52      0.56      0.54      3219
weighted avg       0.93      0.92      0.92      3219


Confusion Matrix

[[  49   18   35]
 [  22   24   49]
 [  66   66 2890]]


In [43]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.905561,0.934625,0.905561,0.918527
1,Bag of Words,SVM,0.920472,0.930176,0.920472,0.924953


## TF-IDF Vectorization

In [44]:
# TF-IDF

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True)

# Fit only on training data

X_train_tfidf = tfidf.fit_transform(X_train)


# Transform test data using the same fitted vectorizer

X_test_tfidf = tfidf.transform(X_test)

# we are checking for the weighted word frequencies

In [45]:
# Checking the Output

print("Training Shape :", X_train_tfidf.shape)
print("Testing Shape  :", X_test_tfidf.shape)

print("Vocabulary Size :", len(tfidf.vocabulary_))

Training Shape : (12873, 10000)
Testing Shape  : (3219, 10000)
Vocabulary Size : 10000


In [46]:
# Train Logistic Regression

lr = LogisticRegression(max_iter=1000, class_weight="balanced",
                        random_state=42)

lr.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.909
Precision: 0.9422
Recall   : 0.909
F1 Score : 0.9232

Classification Report

              precision    recall  f1-score   support

           0       0.38      0.59      0.46       102
           1       0.21      0.43      0.29        95
           2       0.98      0.93      0.96      3022

    accuracy                           0.91      3219
   macro avg       0.53      0.65      0.57      3219
weighted avg       0.94      0.91      0.92      3219


Confusion Matrix

[[  60   28   14]
 [  22   41   32]
 [  75  122 2825]]


In [47]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    class_weight="balanced",
    random_state=42)

svm.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.9258
Precision: 0.9357
Recall   : 0.9258
F1 Score : 0.9304

Classification Report

              precision    recall  f1-score   support

           0       0.41      0.47      0.44       102
           1       0.24      0.33      0.28        95
           2       0.98      0.96      0.97      3022

    accuracy                           0.93      3219
   macro avg       0.54      0.59      0.56      3219
weighted avg       0.94      0.93      0.93      3219


Confusion Matrix

[[  48   21   33]
 [  23   31   41]
 [  46   75 2901]]


In [48]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.905561,0.934625,0.905561,0.918527
1,Bag of Words,SVM,0.920472,0.930176,0.920472,0.924953
2,TF-IDF,Logistic Regression,0.908978,0.942204,0.908978,0.923232
3,TF-IDF,SVM,0.925753,0.935653,0.925753,0.930407


## BERT Embedding

* Dense contextual embeddings
* It takes much longer to generate embeddings.
* It uses more RAM and computation.

Therefore, in this project, only the BERT model implementation is provided as a reference and is not used for the final model comparison.

In [49]:
# Loading the Pre-trained BERT

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [50]:
# BERT Embedding Function

# def get_bert_embeddings(texts):

#    embeddings = []
#    bert_model.eval()
#    with torch.no_grad():

#       for text in texts:

#            encoded = tokenizer(
#                str(text),
#                padding="max_length",
#                truncation=True,
#                max_length=128,
#                return_tensors="pt")

#            output = bert_model(**encoded)

#            cls_embedding = output.last_hidden_state[:,0,:].squeeze().numpy()
#            embeddings.append(cls_embedding)

#    return np.array(embeddings)

In [51]:
# Generate Embeddings

# X_train_bert = get_bert_embeddings(X_train)
# X_test_bert = get_bert_embeddings(X_test)

In [52]:
# Checking the Shape

# print(X_train_bert.shape)
# print(X_test_bert.shape)

In [53]:
# Train Logistic Regression

# lr = LogisticRegression(
#    max_iter=1000,
#    random_state=42)

# lr.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    lr, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"Logistic Regression",
#   "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#   "F1 Score":f1})

In [54]:
# Train SVM

# svm = SVC(
#    kernel="linear",
#    probability=True,
#    random_state=42)

# svm.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    svm, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"SVM",
#    "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#    "F1 Score":f1})

In [55]:
# Display Final Results

# results_df = pd.DataFrame(results)
# results_df

# Deep Learning Model

### Tokenization

In [56]:
# Convert review text into numerical sequences

# Maximum number of words to keep in the vocabulary
MAX_WORDS = 10000

# Maximum number of tokens allowed in each review
MAX_LENGTH = 100

# Create a tokenizer and replace unknown words with <OOV>
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Learn the vocabulary only from the training reviews
tokenizer.fit_on_texts(X_train)

# Convert training reviews into numerical sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)

# Convert test reviews into numerical sequences
# using the vocabulary learned from the training data
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad training sequences to a fixed length
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

# Pad test sequences to the same fixed length
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("X_train_pad shape:", X_train_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

X_train_pad shape: (12873, 100)
X_test_pad shape: (3219, 100)


### Building BiLSTM Model

In [57]:
# Create Sequential model

model = Sequential()

# Word Embedding Layer
# Converts words into dense vector representations

model.add(Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LENGTH))

# Bidirectional LSTM Layer
# Reads the text in both forward and backward directions

model.add(Bidirectional(LSTM(64)))

# Dropout Layer
# Reduces overfitting

model.add(Dropout(0.5))

# Output Layer

model.add(Dense(3, activation='softmax'))     # Multiclass

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [58]:
# Compile the model

model.compile(optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

# Display model summary

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Training the Model

In [59]:
# Get the unique classes
classes = np.unique(y_train)

# Calculate balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

# Convert to dictionary
class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:", class_weight_dict)

Class weights: {np.int64(0): np.float64(10.542997542997544), np.int64(1): np.float64(11.26246719160105), np.int64(2): np.float64(0.3550682664460075)}


In [60]:
# Early stopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)

# Train the model

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight=class_weight_dict)

Epoch 1/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.7207 - loss: 0.9799 - val_accuracy: 0.8194 - val_loss: 0.5875
Epoch 2/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8469 - loss: 0.6631 - val_accuracy: 0.8384 - val_loss: 0.4355
Epoch 3/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8933 - loss: 0.4185 - val_accuracy: 0.8586 - val_loss: 0.3915
Epoch 4/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9288 - loss: 0.2536 - val_accuracy: 0.8882 - val_loss: 0.3130
Epoch 5/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9371 - loss: 0.1928 - val_accuracy: 0.8924 - val_loss: 0.3332
Epoch 6/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9570 - loss: 0.1251 - val_accuracy: 0.8812 - val_loss: 0.3917
Epoch 7/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9558 - loss: 0.1104 - val_accuracy: 0.8870 - val_loss: 0.3471


In [61]:
 # Evaluate the model

loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy) # Predict star ratings

y_pred = model.predict(X_test_pad)

101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8919 - loss: 0.2947
Test Loss : 0.2947191298007965
Test Accuracy : 0.8918918967247009
101/101 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


### Predictions of the BiLSTM model

In [62]:
# Convert probabilities into class labels

y_pred = np.argmax(y_pred, axis=1)  # Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

### Evaluation

In [63]:
# Append LSTM results to the results list

results.append({
    "Embedding": "Tokenizer + Embedding",
    "Model": "LSTM",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1})


### Classification Report

In [64]:
# Classification Report
print(classification_report(y_test, y_pred))
# Confusion Matrix
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.36      0.40      0.38       102
           1       0.16      0.45      0.24        95
           2       0.98      0.92      0.95      3022

    accuracy                           0.89      3219
   macro avg       0.50      0.59      0.52      3219
weighted avg       0.94      0.89      0.91      3219

[[  41   41   20]
 [  16   43   36]
 [  57  178 2787]]


In [65]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.905561,0.934625,0.905561,0.918527
1,Bag of Words,SVM,0.920472,0.930176,0.920472,0.924953
2,TF-IDF,Logistic Regression,0.908978,0.942204,0.908978,0.923232
3,TF-IDF,SVM,0.925753,0.935653,0.925753,0.930407
4,Tokenizer + Embedding,LSTM,0.891892,0.936549,0.891892,0.911360


# Save the Model

In [81]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    svm,
    "../models/recipe_sentiment_svm.pkl"
)

print("SVM model saved successfully!")

SVM model saved successfully!


In [83]:
from sklearn.pipeline import Pipeline
import joblib
import os

# Combine your existing TF-IDF vectorizer and trained SVM
final_model = Pipeline([
    ("tfidf", tfidf),
    ("svm", svm)
])

print("Final TF-IDF + SVM pipeline created!")


Final TF-IDF + SVM pipeline created!


In [93]:
import os
import joblib

# Project's models folder
models_path = "/content/C:\\Users\\hp\\Desktop\\AIML\\NLP_Project_Receipe_Reviews_and_Feedback/models"

# Create folder if it doesn't exist
os.makedirs(models_path, exist_ok=True)

# File path
model_path = os.path.join(
    models_path,
    "recipe_sentiment_model.pkl"
)

# Save model automatically
joblib.dump(final_model, model_path)

print("✅ Model saved automatically!")
print(f"📁 Location: {model_path}")
print(f"📦 File size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")


✅ Model saved automatically!
📁 Location: /content/C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback/models/recipe_sentiment_model.pkl
📦 File size: 1.97 MB


In [91]:
loaded_model = joblib.load(model_path)

print("Model loaded successfully!")
print(loaded_model)


Model loaded successfully!
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=10000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('svm',
                 SVC(class_weight='balanced', kernel='linear', probability=True,
                     random_state=42))])


In [94]:
from google.colab import files

files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [92]:
test_reviews = [
    "This recipe was absolutely delicious!",
    "The recipe was okay, nothing special.",
    "I hated this recipe. It was terrible."
]

predictions = loaded_model.predict(test_reviews)

for review, prediction in zip(test_reviews, predictions):
    print("Review:", review)
    print("Prediction:", prediction)
    print("-" * 50)


Review: This recipe was absolutely delicious!
Prediction: 2
--------------------------------------------------
Review: The recipe was okay, nothing special.
Prediction: 1
--------------------------------------------------
Review: I hated this recipe. It was terrible.
Prediction: 0
--------------------------------------------------


In [66]:
# Get the project root directory
BASE_DIR = Path.cwd().parent

# Create models folder automatically
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Models directory:", MODEL_DIR)

Project directory: /
Models directory: /models


In [67]:
# SAVE LSTM MODEL AUTOMATICALLY

LSTM_MODEL_PATH = MODEL_DIR / "lstm_model.keras"

model.save(
    LSTM_MODEL_PATH
)

print("\nLSTM model saved successfully:")
print(LSTM_MODEL_PATH)


LSTM model saved successfully:
/models/lstm_model.keras


In [68]:
# SAVE TOKENIZER AUTOMATICALLY

TOKENIZER_PATH = MODEL_DIR / "tokenizer.pkl"

with open(
    TOKENIZER_PATH,
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )

print("\nTokenizer saved successfully:")
print(TOKENIZER_PATH)


Tokenizer saved successfully:
/models/tokenizer.pkl


In [69]:
# Label mapping used by the model
id_to_label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

print(id_to_label)

{0: 'Negative', 1: 'Neutral', 2: 'Positive'}


In [70]:
# SAVE LABEL MAPPING

LABEL_MAPPING_PATH = MODEL_DIR / "label_mapping.pkl"

with open(
    LABEL_MAPPING_PATH,
    "wb"
) as file:

    pickle.dump(
        id_to_label,
        file
    )

print("\nLabel mapping saved successfully:")
print(LABEL_MAPPING_PATH)


Label mapping saved successfully:
/models/label_mapping.pkl


In [71]:
# SAVE MODEL CONFIGURATION

# Number of output classes
NUM_CLASSES = len(np.unique(y_train))

# Model configuration
CONFIG = {
    "max_words": MAX_WORDS,
    "max_length": MAX_LENGTH,
    "num_classes": NUM_CLASSES
}

# Configuration file path
CONFIG_PATH = MODEL_DIR / "config.pkl"

# Save configuration
with open(CONFIG_PATH, "wb") as file:
    pickle.dump(CONFIG, file)

print("\nModel configuration saved successfully:")
print(CONFIG_PATH)

print("\nConfiguration:")
print(CONFIG)


Model configuration saved successfully:
/models/config.pkl

Configuration:
{'max_words': 10000, 'max_length': 100, 'num_classes': 3}


In [72]:
# FINAL CHECK

print("\n======================================")
print("ALL MODEL FILES SAVED SUCCESSFULLY")
print("======================================")

for file in MODEL_DIR.iterdir():

    print(file.name)


ALL MODEL FILES SAVED SUCCESSFULLY
lstm_model.keras
label_mapping.pkl
tokenizer.pkl
config.pkl


In [73]:
from pathlib import Path
import shutil

# Your Windows project location
PROJECT_DIR = Path(r"C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback")
PROJECT_MODELS = PROJECT_DIR / "models"

PROJECT_MODELS.mkdir(parents=True, exist_ok=True)

# Files created by your notebook
files = [
    "lstm_model.keras",
    "tokenizer.pkl",
    "label_mapping.pkl",
    "config.pkl"
]

for filename in files:
    source = Path("/models") / filename
    destination = PROJECT_MODELS / filename

    if source.exists():
        shutil.copy2(source, destination)
        print(f"Copied: {filename}")
    else:
        print(f"NOT FOUND: {source}")

print("\nProject models folder:")
for file in PROJECT_MODELS.iterdir():
    print(file)


Copied: lstm_model.keras
Copied: tokenizer.pkl
Copied: label_mapping.pkl
Copied: config.pkl

Project models folder:
C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback/models/lstm_model.keras
C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback/models/label_mapping.pkl
C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback/models/tokenizer.pkl
C:\Users\hp\Desktop\AIML\NLP_Project_Receipe_Reviews_and_Feedback/models/config.pkl


In [74]:
from google.colab import files

files.download("/models/lstm_model.keras")
files.download("/models/tokenizer.pkl")
files.download("/models/label_mapping.pkl")
files.download("/models/config.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>